# Hearing Reality — Colab Main Runner

End-to-end notebook for the *Two-Stream Deepfake Audio Detection* thesis.
Run cells top-to-bottom. Only **Section 1 — Path Configuration** (cell 4) needs manual edits.

| Section | What it does |
|---|---|
| 1 | Session init: install, Drive mount, path config, GPU check |
| 2 | Fit or load Stream 2 PCA |
| 3 | Train Setup A / B / C (skips if checkpoint exists) |
| 4 | Extract scores for clean + 6 codec conditions |
| 5 | Print 3×7 EER% and min-tDCF results tables |
| 6 | Measure Real-Time Factor (RTF) per setup |

## Section 1 — Session Initialisation

In [ ]:
# ── 1.1  Clone / update repo and install package ────────────────────────────
import os, subprocess, sys

REPO_URL = "https://github.com/Remigaraki/TwoStream-Audio-Forensics.git"
REPO_DIR = "/content/TwoStream-Audio-Forensics"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=True)
print("Install complete. CWD:", os.getcwd())

In [ ]:
# ── 1.2  Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.isdir("/content/drive/MyDrive"), "Drive mount failed"
print("Drive mounted successfully.")

In [ ]:
# ── 1.3  PATH CONFIGURATION — edit this cell ─────────────────────────────────
from pathlib import Path

# Root of the ASVspoof 5 dataset on your Drive
DATA_DIR = "/content/drive/MyDrive/ASVspoof5"

# Protocol files
PROTOCOL_TRAIN      = f"{DATA_DIR}/protocols/train.txt"
PROTOCOL_EVAL_CLEAN = f"{DATA_DIR}/protocols/eval.txt"

# Where to save checkpoints and PCA
CHECKPOINT_DIR = f"{DATA_DIR}/checkpoints"

# Where score CSVs are written
RESULTS_DIR = f"{DATA_DIR}/results"

# Root for codec-tortured eval audio produced by torture_pipeline
# Subdirs: opus_16k/, opus_32k/, opus_64k/, mp3_64k/, mp3_128k/, aac_128k/
TORTURED_BASE = f"{DATA_DIR}/tortured_eval"

# Training hyper-parameters
EPOCHS     = 50
BATCH_SIZE = 32
LR         = 1e-4
SEED       = 42

# W&B logging — set USE_WANDB=True to enable
USE_WANDB     = False
WANDB_PROJECT = "twostream-audio-forensics"

# Which experimental setups to run (any subset of ["A", "B", "C"])
SETUPS_TO_RUN = ["A", "B", "C"]

# ─────────────────────────────────────────────────────────────────────────────
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
print("Paths configured.")
for label, val in [("DATA_DIR", DATA_DIR), ("CHECKPOINT_DIR", CHECKPOINT_DIR),
                   ("RESULTS_DIR", RESULTS_DIR), ("SETUPS_TO_RUN", SETUPS_TO_RUN)]:
    print(f"  {label} = {val}")

In [ ]:
# ── 1.4  GPU check ───────────────────────────────────────────────────────────
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")

## Section 2 — Stream 2 PCA Fitting

Fits the bispectrum PCA basis on the training set and saves it to `CHECKPOINT_DIR/pca.pkl`.  
Re-running this cell is a no-op if the file already exists.

In [ ]:
from src.stream2.extract_bispectrum import Stream2ForensicFeaturePipeline
from data.dataset_loader import fit_stream2_pipeline_from_protocol

PCA_PATH = Path(CHECKPOINT_DIR) / "pca.pkl"

if PCA_PATH.exists():
    print(f"PCA already exists — loading from {PCA_PATH}")
    pipeline = Stream2ForensicFeaturePipeline.load(PCA_PATH)
else:
    print(f"Fitting Stream 2 PCA on up to 2000 samples …")
    pipeline = fit_stream2_pipeline_from_protocol(
        base_dir=DATA_DIR,
        protocol_file=PROTOCOL_TRAIN,
        max_items=2000,
        random_seed=SEED,
    )
    pipeline.save(PCA_PATH)
    print(f"PCA saved to {PCA_PATH}")

assert pipeline.is_fitted, "Pipeline is not fitted — check DATA_DIR and PROTOCOL_TRAIN"
print(f"Pipeline ready. n_components={pipeline.pca.n_components_}")

## Section 3 — Training

Trains each setup listed in `SETUPS_TO_RUN`.  
If a checkpoint already exists for a setup, training is skipped — delete the `.pt` file manually to retrain.

In [ ]:
# ── 3.1  Imports ─────────────────────────────────────────────────────────────
import math

from src.train import build_model
from src.training.train_loop import (
    TrainingConfig,
    build_two_stream_loaders,
    fit_two_stream_model,
    set_seed,
)
from data.dataset_loader import ASVspoofDataset
from src.utils.logger import setup_logger

set_seed(SEED)
print("Training utilities imported.")

In [ ]:
# ── 3.2  Training loop ───────────────────────────────────────────────────────
training_histories = {}

for setup in SETUPS_TO_RUN:
    checkpoint_path = Path(CHECKPOINT_DIR) / f"best_setup_{setup}.pt"

    if checkpoint_path.exists():
        print(f"\n[Setup {setup}] Checkpoint exists — skipping training.")
        training_histories[setup] = None
        continue

    print(f"\n{'='*60}")
    print(f"[Setup {setup}] Building dataset …")

    train_dataset = ASVspoofDataset(
        base_dir=DATA_DIR,
        protocol_file=PROTOCOL_TRAIN,
        stream2_pipeline=pipeline,
        return_stream2=True,
        training=True,
    )
    print(f"[Setup {setup}] Dataset: {len(train_dataset)} samples")

    train_loader, val_loader = build_two_stream_loaders(
        train_dataset,
        batch_size=BATCH_SIZE,
        val_fraction=0.2,
        seed=SEED,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    model = build_model(setup)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"[Setup {setup}] Model: {n_params / 1e6:.2f}M parameters")

    log_dir = Path(CHECKPOINT_DIR) / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = setup_logger(
        f"train_setup_{setup}",
        str(log_dir / f"setup_{setup}.log"),
    )

    config = TrainingConfig(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        weight_decay=1e-4,
        checkpoint_path=str(checkpoint_path),
        log_dir=str(log_dir),
        scheduler_type="cosine",
        seed=SEED,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
        use_wandb=USE_WANDB,
        wandb_project=WANDB_PROJECT,
        wandb_run_name=f"setup_{setup}",
    )

    print(f"[Setup {setup}] Training for {EPOCHS} epochs …")
    history = fit_two_stream_model(model, train_loader, val_loader, config, logger=logger)
    training_histories[setup] = history

    valid_eers = [e["val_eer"] for e in history if not math.isnan(e["val_eer"])]
    best_eer = min(valid_eers) if valid_eers else float("nan")
    print(f"[Setup {setup}] Done. Best val EER: {best_eer:.4f}")
    print(f"[Setup {setup}] Checkpoint: {checkpoint_path}")

print("\nAll training complete.")

## Section 4 — Score Extraction

Runs inference on the **clean** eval split and all **6 codec conditions** for each setup.  
Writes one CSV per (setup, condition) pair to `RESULTS_DIR`.  
Skips any CSV that already exists.

In [ ]:
# ── 4.1  Define conditions ───────────────────────────────────────────────────
from src.eval.score_extractor import extract_scores

# Labels must match subdirectory names produced by data.torture_pipeline.run_all_conditions
CODEC_CONDITIONS = [
    "opus_16k",
    "opus_32k",
    "opus_64k",
    "mp3_64k",
    "mp3_128k",
    "aac_128k",
]

# (condition_label, audio_root, protocol_file)
# Codec-tortured dirs share the same protocol — file tokens are identical,
# only the audio root changes.
ALL_EVAL_CONDITIONS = [("clean", DATA_DIR, PROTOCOL_EVAL_CLEAN)] + [
    (label, str(Path(TORTURED_BASE) / label), PROTOCOL_EVAL_CLEAN)
    for label in CODEC_CONDITIONS
]

print(f"Eval conditions ({len(ALL_EVAL_CONDITIONS)}):")
for label, root, _ in ALL_EVAL_CONDITIONS:
    exists = Path(root).exists()
    print(f"  {label:<12}  {'OK' if exists else 'MISSING'}  {root}")

In [ ]:
# ── 4.2  Score extraction loop ───────────────────────────────────────────────
score_csv_map = {}  # {(setup, condition): csv_path}

for setup in SETUPS_TO_RUN:
    checkpoint_path = Path(CHECKPOINT_DIR) / f"best_setup_{setup}.pt"
    if not checkpoint_path.exists():
        print(f"[Setup {setup}] No checkpoint found — skipping score extraction.")
        continue

    for condition_label, eval_data_dir, eval_protocol in ALL_EVAL_CONDITIONS:
        out_csv = Path(RESULTS_DIR) / f"scores_{setup}_{condition_label}.csv"
        score_csv_map[(setup, condition_label)] = str(out_csv)

        if out_csv.exists():
            print(f"[{setup}/{condition_label}] CSV exists — skipping.")
            continue

        if not Path(eval_data_dir).exists():
            print(f"[{setup}/{condition_label}] Audio dir missing — skipping.")
            continue

        print(f"[{setup}/{condition_label}] Extracting scores …")
        extract_scores(
            setup=setup,
            checkpoint_path=str(checkpoint_path),
            data_dir=str(eval_data_dir),
            protocol_path=str(eval_protocol),
            pca_path=str(PCA_PATH),
            output_csv=str(out_csv),
            batch_size=BATCH_SIZE,
            num_workers=2,
        )
        print(f"  -> {out_csv}")

print("\nScore extraction complete.")

## Section 5 — Evaluation Results Table

Loads all score CSVs and prints a **3 × 7** results table (3 setups × 7 conditions).  
Metrics: EER% (lower is better) and min-tDCF (lower is better).

In [ ]:
import csv
import numpy as np
from src.utils.metrics import compute_eer, compute_t_dcf

CONDITION_ORDER = ["clean"] + CODEC_CONDITIONS

results = {}  # {(setup, condition): {"eer": float, "tdcf": float}}

for setup in SETUPS_TO_RUN:
    for cond in CONDITION_ORDER:
        csv_path = score_csv_map.get((setup, cond))
        if csv_path is None or not Path(csv_path).exists():
            results[(setup, cond)] = {"eer": float("nan"), "tdcf": float("nan")}
            continue

        y_true, y_score = [], []
        with open(csv_path, newline="", encoding="utf-8") as fh:
            for row in csv.DictReader(fh):
                y_true.append(int(row["label"]))
                y_score.append(float(row["score"]))

        y_true  = np.array(y_true)
        y_score = np.array(y_score)

        try:
            eer,  _ = compute_eer(y_true, y_score)
            tdcf, _ = compute_t_dcf(y_true, y_score)
        except Exception as exc:
            print(f"  WARNING [{setup}/{cond}]: {exc}")
            eer, tdcf = float("nan"), float("nan")

        results[(setup, cond)] = {"eer": eer, "tdcf": tdcf}

# ── Print tables ─────────────────────────────────────────────────────────────
COL_W = 12
header = f"{'':8}" + "".join(f"{c:>{COL_W}}" for c in CONDITION_ORDER)
sep    = "-" * len(header)

print("\n" + "=" * len(header))
print("EER (%) — lower is better")
print("=" * len(header))
print(header)
print(sep)
for setup in SETUPS_TO_RUN:
    row = f"Setup {setup}  "
    for cond in CONDITION_ORDER:
        v = results.get((setup, cond), {}).get("eer", float("nan"))
        row += f"{v*100:>{COL_W}.2f}" if not np.isnan(v) else f"{'N/A':>{COL_W}}"
    print(row)

print("\n" + "=" * len(header))
print("min-tDCF — lower is better")
print("=" * len(header))
print(header)
print(sep)
for setup in SETUPS_TO_RUN:
    row = f"Setup {setup}  "
    for cond in CONDITION_ORDER:
        v = results.get((setup, cond), {}).get("tdcf", float("nan"))
        row += f"{v:>{COL_W}.4f}" if not np.isnan(v) else f"{'N/A':>{COL_W}}"
    print(row)
print("=" * len(header))

## Section 6 — Real-Time Factor (RTF) Measurement

Times a single forward pass for each setup and computes `RTF = inference_time / audio_duration`.  
**Target: RTF < 1.0** for real-time capable deployment.

In [ ]:
import time

AUDIO_DURATION_S = 4.0       # process_waveform clips/pads to 64000 samples @ 16 kHz
AUDIO_SAMPLES    = 64000
STATS_DIM        = 248       # 120 LFCC + 128 bispectrum PCA components
N_WARMUP         = 5
N_TIMED          = 20

print(f"Audio: {AUDIO_DURATION_S}s ({AUDIO_SAMPLES} samples)  |  "
      f"Warmup: {N_WARMUP}  |  Timed runs: {N_TIMED}\n")

rtf_results = {}

for setup in SETUPS_TO_RUN:
    rtf_model = build_model(setup)
    checkpoint_path = Path(CHECKPOINT_DIR) / f"best_setup_{setup}.pt"

    if checkpoint_path.exists():
        ckpt = torch.load(str(checkpoint_path), map_location=device)
        rtf_model.load_state_dict(ckpt["model_state_dict"])
        print(f"[Setup {setup}] Loaded checkpoint weights.")
    else:
        print(f"[Setup {setup}] No checkpoint — timing random weights.")

    rtf_model = rtf_model.to(device).eval()

    dummy_audio = torch.randn(1, 1, AUDIO_SAMPLES, device=device)
    dummy_stats = torch.randn(1, STATS_DIM, device=device)

    with torch.no_grad():
        for _ in range(N_WARMUP):
            rtf_model(dummy_audio, dummy_stats)
        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        for _ in range(N_TIMED):
            rtf_model(dummy_audio, dummy_stats)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

    avg_ms = elapsed / N_TIMED * 1000
    rtf    = (elapsed / N_TIMED) / AUDIO_DURATION_S
    rtf_results[setup] = rtf
    status = "real-time capable" if rtf < 1.0 else "SLOWER THAN REAL-TIME"
    print(f"  Setup {setup}: avg={avg_ms:.2f} ms  RTF={rtf:.4f}  ({status})")

print("\nRTF Summary:")
for setup, rtf in rtf_results.items():
    print(f"  Setup {setup}: RTF = {rtf:.4f}")